We use OBR's own forecast of housing starts/completions since it embeds the policy scenario

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
import json
import openpyxl
from python.functions.bridge import parse_quarter, build_bridge_inputs, run_bridge

In [ ]:
RAW_DIR = "../../data/raw"
OUT_DIR = "../../data/outputs"

B = build_bridge_inputs()

wb = openpyxl.load_workbook(
    f"{RAW_DIR}/OBR/efo-march-2026-detailed-forecast-tables-economy.xlsx",
    read_only=True, data_only=True
)
ws = wb["1.16"]
rows = list(ws.iter_rows(values_only=True))

recs = [(r[1], r[5]) for r in rows if r[1] and len(str(r[1])) == 6 and str(r[1])[4] == "Q"]
obr = pd.DataFrame(recs, columns=["period", "starts_uk"])
obr["period"] = pd.PeriodIndex(obr["period"], freq="Q")
obr = obr[obr["period"] >= "2026Q1"]  # check this matches your ensemble/VECM CSVs' start quarter

ENGLAND_SHARE = 0.846
obr["starts_eng"] = obr["starts_uk"] * ENGLAND_SHARE
obr["ensemble_log"] = np.log(obr["starts_eng"])
obr_out = obr[["period", "ensemble_log"]].copy()
obr_out["period"] = obr_out["period"].astype(str)

obr_out.to_csv(f"{OUT_DIR}/OBR/obr_own_starts_scenario.csv", index=False)
obr_out.head()

In [ ]:
rdl  = run_bridge(f"{OUT_DIR}/forecasts/obr_scenario_forecasts.csv", "ensemble_log",
                   B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"])
vecm = run_bridge(f"{OUT_DIR}/forecasts/vecm_unconditional_forecast.csv", "vecm_log",
                   B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"], strip_space=True)
obr_reform = run_bridge(f"{OUT_DIR}/OBR/obr_own_starts_scenario.csv", "ensemble_log",
                         B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"])

target = 1_500_000
for name, d in [("ARDL/NARDL ensemble (baseline)", rdl),
                ("VECM (unconditional)", vecm),
                ("OBR own starts forecast (reform-inclusive)", obr_reform)]:
    print(name)
    for fy, v in d.items():
        print(f"{fy}: {v:,.0f}")
    cumulative = sum(d.values())
    print(f"Cumulative: {cumulative:,.0f} ({100*cumulative/target:.1f}% of {target:,}, "
          f"shortfall {target-cumulative:,.0f})\n")

In [ ]:
import matplotlib.pyplot as plt

fy_start = lambda s: int(str(s)[:4])  # "2022-23" -> 2022

# Actual history from LT120, plus the 2024-25 actual already in delivery
actual = B["lt120"]["Total net additional dwellings"].dropna()
actual.index = actual.index.map(fy_start)
if 2024 not in actual.index:
    actual.loc[2024] = rdl["2024-25"]
actual = actual.sort_index()

# Forecast path (2025-26 onward)
def fcast_series(d):
    s = pd.Series({fy_start(k): v for k, v in d.items() if fy_start(k) >= 2025}).sort_index()
    return pd.concat([actual.iloc[[-1]], s])   # prepend last actual to close the gap

join_rdl        = fcast_series(rdl)
join_vecm       = fcast_series(vecm)
join_obr_reform = fcast_series(obr_reform)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(actual.index, actual.values, color="#1f4e79", lw=2, label="Actual")
ax.plot(join_rdl.index,  join_rdl.values,  color="#c0392b", lw=2, ls="--",
        marker="o", label="ARDL/NARDL ensemble (OBR-conditioned)")
ax.plot(join_vecm.index, join_vecm.values, color="#e08e0b", lw=2, ls=":",
        marker="s", label="VECM (unconditional system)")
ax.plot(join_obr_reform.index, join_obr_reform.values, color="#2e8b57", lw=2, ls="-.",
        marker="^", label="OBR own starts (reform-inclusive)")
ax.set_ylabel("Net additional dwellings")
ax.set_xlabel("Financial year (start)")
ax.legend()
plt.tight_layout()
plt.show()